# 🔍 Retrieval Augmented Generation (RAG)
## Lezione pratica con Chroma e Ollama

---

### Cos'è il RAG?

**RAG (Retrieval Augmented Generation)** è un'architettura che combina due componenti:

1. **Retriever**: recupera i documenti più rilevanti da una knowledge base in base alla query dell'utente
2. **Generator**: un LLM che genera la risposta usando i documenti recuperati come contesto

```
┌─────────────┐     ┌─────────────────┐     ┌──────────────┐
│   Query     │────▶│   Retriever     │────▶│   Context    │
│  Utente     │     │ (Vector Search) │     │  Documenti   │
└─────────────┘     └─────────────────┘     └──────┬───────┘
                                                   │
                                           ┌───────▼───────┐
                                           │  LLM (Ollama) │
                                           │  + Prompt     │
                                           └───────┬───────┘
                                                   │
                                           ┌───────▼───────┐
                                           │   Risposta    │
                                           └───────────────┘
```

**Strumenti usati in questa lezione:**
- **[ChromaDB](https://www.trychroma.com/)**: database vettoriale in-memory per lo storage e il retrieval dei documenti
- **[Ollama](https://ollama.ai/)**: LLM locale (non serve cloud, tutto gira sulla tua macchina)
- **[Sentence Transformers](https://www.sbert.net/)**: modello per creare gli embedding dei testi


---
## ⚙️ SETUP - Ambiente virtuale e dipendenze

Prima di tutto, esegui i seguenti comandi nel terminale per configurare l'ambiente.

### 1. Prerequisiti: installare Ollama

Scarica e installa Ollama da: https://ollama.ai/download

Poi, dal terminale, scarica il modello che useremo:
```bash
ollama pull llama3.2
```
Avvia il server Ollama (se non è già in esecuzione):
```bash
ollama serve
```

### 2. Creare il virtual environment

```bash
# Crea il virtual environment
python -m venv venv_rag

# Attivalo (macOS/Linux)
source venv_rag/bin/activate

# Attivalo (Windows)
# venv_rag\Scripts\activate
```

### 3. Installare i pacchetti

```bash
pip install chromadb sentence-transformers ollama
```

Oppure puoi eseguire la cella qui sotto direttamente dal notebook:

In [1]:
# Installa le dipendenze necessarie
# (esegui solo se non hai già installato i pacchetti nel virtualenv)
%pip install chromadb sentence-transformers ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
models

NameError: name 'models' is not defined

In [3]:
# Verifica che Ollama sia raggiungibile
import ollama

try:
    models = ollama.list()
    print("✅ Ollama è attivo e raggiungibile.")
    print("Modelli disponibili:")
    for model in models.get('models', []):
        print(f"  - {model}")
except Exception as e:
    print(f"❌ Ollama non raggiungibile: {e}")
    print("Assicurati che 'ollama serve' sia in esecuzione nel terminale.")

✅ Ollama è attivo e raggiungibile.
Modelli disponibili:
  - model='nomic-embed-text:latest' modified_at=datetime.datetime(2026, 9, 30, 9, 42, 53, 505249, tzinfo=TzInfo(7200)) digest='0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f' size=274302450 details=ModelDetails(parent_model='', format='gguf', family='nomic-bert', families=['nomic-bert'], parameter_size='137M', quantization_level='F16')
  - model='qwen3.5:9b' modified_at=datetime.datetime(2026, 9, 29, 10, 43, 11, 575785, tzinfo=TzInfo(7200)) digest='6488c96fa5faab64bb65cbd30d4289e20e6130ef535a93ef9a49f42eda893ea7' size=6594474711 details=ModelDetails(parent_model='', format='gguf', family='qwen35', families=['qwen35'], parameter_size='9.7B', quantization_level='Q4_K_M')
  - model='qwen2.5-coder:7b' modified_at=datetime.datetime(2026, 9, 29, 10, 27, 53, 162909, tzinfo=TzInfo(7200)) digest='dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e75ed87571486f4364' size=4683087561 details=ModelDetails(parent_model='', format=

---
# 📦 ESEMPIO 1 — RAG Semplice
## Chatbot aziendale con knowledge base in-memory

**Scenario:** Un dipendente di TechSphere S.r.l. può fare domande sulle policy aziendali.
La knowledge base è composta da una lista di frasi con informazioni aziendali.

**Flusso:**
```
Frasi aziendali ──▶ Embedding ──▶ ChromaDB (in-memory)
                                        │
Query utente ──▶ Embedding ──────▶ Similarity Search
                                        │
                                   Top-K Documenti
                                        │
                                   Prompt + Contesto
                                        │
                                     Ollama LLM
                                        │
                                      Risposta
```

### 1.1 — Creare la Knowledge Base (frasi aziendali)

In [4]:
# Informazioni aziendali come lista di frasi
# In un caso reale queste potrebbero venire da un DB, un CMS, ecc.

knowledge_base = [
    # Profilo aziendale
    "TechSphere S.r.l. è un'azienda italiana fondata nel 2010 con sede principale a Milano.",
    "TechSphere opera nel settore dello sviluppo software, della consulenza IT e dei servizi cloud.",
    "TechSphere conta attualmente 350 dipendenti distribuiti in quattro sedi: Milano, Roma, Torino e Bologna.",
    "Il fatturato annuo di TechSphere per il 2023 è stato di 42 milioni di euro, con una crescita del 18%.",

    # Prodotti
    "SphereERP è il gestionale ERP per PMI di TechSphere, disponibile cloud e on-premise al costo di 299 euro al mese.",
    "SphereCRM è il CRM aziendale con intelligenza artificiale per la gestione dei clienti, al costo di 199 euro al mese.",
    "SphereHR è la piattaforma per la gestione delle risorse umane che include moduli per paghe, presenze e formazione, al costo di 149 euro al mese.",
    "SphereAnalytics è la piattaforma di business intelligence con dashboard personalizzabili, al costo di 249 euro al mese.",

    # Benefit e HR
    "Ogni dipendente TechSphere ha diritto a 26 giorni di ferie all'anno.",
    "TechSphere prevede lo smart working fino a 3 giorni a settimana per tutti i ruoli compatibili.",
    "Ogni dipendente ha un budget annuo di formazione di 1.500 euro e accesso gratuito a Coursera, Udemy Business e Pluralsight.",
    "I dipendenti TechSphere ricevono ticket restaurant del valore di 8 euro al giorno.",
    "TechSphere offre assicurazione sanitaria integrativa per il dipendente e i familiari a carico.",
    "Il piano di welfare aziendale Edenred vale 800 euro all'anno per ogni dipendente.",
    "I dipendenti senior con oltre 3 anni di anzianità hanno accesso allo stock option plan.",

    # Processi e metodologie
    "TechSphere adotta la metodologia Agile con framework Scrum per tutti i progetti di sviluppo.",
    "Gli sprint di sviluppo hanno durata di 2 settimane.",
    "Il tool principale per il project management è Jira; per la comunicazione interna si usa Slack.",
    "La valutazione delle performance avviene due volte l'anno, a giugno e a dicembre, con il modello OKR.",
    "Il bonus annuale può arrivare fino al 15% della retribuzione annua lorda per i dipendenti con performance eccellente.",

    # Sicurezza e compliance
    "TechSphere è certificata ISO/IEC 27001:2022 per la gestione della sicurezza delle informazioni.",
    "Tutti i dati dei clienti sono trattati in conformità con il GDPR (Regolamento UE 2016/679).",
    "Tutti i dipendenti devono completare annualmente il corso obbligatorio di Security Awareness.",

    # Contatti
    "La sede principale di TechSphere è in Via della Innovazione 42, 20121 Milano.",
    "Il numero di centralino di TechSphere è +39 02 9876 5432.",
    "L'email HR di TechSphere è hr@techsphere.it per richieste relative alle risorse umane.",
    "Il sito web aziendale di TechSphere è www.techsphere.it.",
]

print(f"✅ Knowledge base creata con {len(knowledge_base)} frasi.")

✅ Knowledge base creata con 27 frasi.


### 1.2 — Inizializzare ChromaDB e inserire i documenti

In [5]:
import chromadb
from chromadb.utils import embedding_functions

# 1. Crea un client ChromaDB in-memory (nessun file su disco)
client = chromadb.Client()

# 2. Definisce la funzione di embedding
#    SentenceTransformerEmbeddingFunction usa un modello locale per generare i vettori
#    'paraphrase-multilingual-MiniLM-L12-v2' supporta l'italiano
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# 3. Crea (o ottieni) la collection — analogo a una tabella in un DB relazionale
collection = client.get_or_create_collection(
    name="techsphere_kb",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # usa la distanza coseno per la similarity
)

# 4. Inserisce i documenti nella collection
#    ChromaDB calcola automaticamente l'embedding di ogni documento
collection.add(
    documents=knowledge_base,
    ids=[f"doc_{i}" for i in range(len(knowledge_base))]
)

print(f"✅ Collection '{collection.name}' creata.")
print(f"   Documenti indicizzati: {collection.count()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Collection 'techsphere_kb' creata.
   Documenti indicizzati: 27


### 1.3 — Funzione di Retrieval

Il retrieval consiste nel trovare i documenti più simili alla query dell'utente usando la **similarità coseno** tra i vettori di embedding.

In [6]:
def retrieve(query: str, n_results: int = 3) -> list[str]:
    """
    Recupera i documenti più rilevanti dalla knowledge base.
    
    Args:
        query: la domanda dell'utente
        n_results: numero di documenti da recuperare
    
    Returns:
        lista di stringhe con i documenti più rilevanti
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    # results['documents'] è una lista di liste (una per ogni query)
    return results['documents'][0]


# Test del retrieval
test_query = "Quante ferie ho all'anno?"
retrieved_docs = retrieve(test_query)

print(f"Query: '{test_query}'")
print(f"\nDocumenti recuperati ({len(retrieved_docs)}):")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  [{i}] {doc}")

Query: 'Quante ferie ho all'anno?'

Documenti recuperati (3):
  [1] Ogni dipendente TechSphere ha diritto a 26 giorni di ferie all'anno.
  [2] Ogni dipendente ha un budget annuo di formazione di 1.500 euro e accesso gratuito a Coursera, Udemy Business e Pluralsight.
  [3] Il fatturato annuo di TechSphere per il 2023 è stato di 42 milioni di euro, con una crescita del 18%.


### 1.4 — Funzione di Generation con Ollama

Costruiamo il prompt che passa il contesto recuperato all'LLM e otteniamo la risposta.

In [7]:
import ollama

# Modello da usare — assicurati di averlo scaricato con: ollama pull llama3.2
OLLAMA_MODEL = "llama3.2"


def generate_answer(query: str, context_docs: list[str]) -> str:
    """
    Genera una risposta usando Ollama con il contesto recuperato.
    
    Args:
        query: la domanda dell'utente
        context_docs: lista di documenti recuperati dal retriever
    
    Returns:
        la risposta generata dall'LLM
    """
    # Costruisce il contesto come stringa unica
    context = "\n".join([f"- {doc}" for doc in context_docs])

    # Prompt strutturato: system + user
    prompt = f"""Sei un assistente virtuale aziendale di TechSphere S.r.l.
Rispondi alle domande dei dipendenti basandoti ESCLUSIVAMENTE sulle informazioni fornite nel contesto.
Se l'informazione non è presente nel contesto, dì chiaramente che non lo sai.
Rispondi sempre in italiano, in modo chiaro e professionale.

CONTESTO:
{context}

DOMANDA: {query}

RISPOSTA:"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response['message']['content']


print("✅ Funzione generate_answer definita.")

✅ Funzione generate_answer definita.


### 1.5 — Pipeline RAG completa

Combiniamo retrieval e generation in un'unica funzione.

In [8]:
def rag_chatbot(query: str, n_results: int = 3, verbose: bool = True) -> str:
    """
    Pipeline RAG completa: retrieve + generate.
    
    Args:
        query: la domanda dell'utente
        n_results: numero di documenti da recuperare
        verbose: se True, mostra i documenti recuperati
    
    Returns:
        la risposta finale del chatbot
    """
    print(f"\n{'='*60}")
    print(f"📩 Domanda: {query}")
    print(f"{'='*60}")

    # Step 1: RETRIEVE
    context_docs = retrieve(query, n_results=n_results)

    if verbose:
        print(f"\n🔍 Documenti recuperati ({len(context_docs)}):")
        for i, doc in enumerate(context_docs, 1):
            print(f"   [{i}] {doc}")

    # Step 2: GENERATE
    print("\n⏳ Generazione risposta in corso...")
    answer = generate_answer(query, context_docs)

    print(f"\n🤖 Risposta:\n{answer}")
    print(f"{'='*60}\n")

    return answer

In [9]:
# 🧪 Test con alcune domande
domande = [
    "Quante ferie ho diritto ogni anno?",
    "Posso lavorare da casa? Quanti giorni alla settimana?",
    "Quanto costa SphereERP?",
    "Come funziona il bonus di fine anno?",
]

for domanda in domande:
    rag_chatbot(domanda, verbose=False)


📩 Domanda: Quante ferie ho diritto ogni anno?

⏳ Generazione risposta in corso...

🤖 Risposta:
Puoi avere diritto a 26 giorni di ferie all'anno.


📩 Domanda: Posso lavorare da casa? Quanti giorni alla settimana?

⏳ Generazione risposta in corso...

🤖 Risposta:
Ciao! Sono felice di aiutarti.

Purtroppo, non ho informazioni specifiche sul tuo ruolo lavorativo all'interno di TechSphere. Tuttavia, posso dirti che la nostra politica lo smart working prevede che i dipendenti lavorino fino a 3 giorni a settimana. Per quanto riguarda il luogo di lavoro, puoi scegliere di lavorare da casa o nel nostro ufficio, a seconda delle tue esigenze e preferenze.

Se hai dubbi o bisogno di ulteriori informazioni, non esitare a chiedere!


📩 Domanda: Quanto costa SphereERP?

⏳ Generazione risposta in corso...

🤖 Risposta:
Il prezzo di SphereERP è di 299 euro al mese.


📩 Domanda: Come funziona il bonus di fine anno?

⏳ Generazione risposta in corso...

🤖 Risposta:
Il bonus di fine anno, come stabilito, è 

---
# 📄 ESEMPIO 2 — RAG Avanzato
## Chatbot aziendale con lettura da file, cleaning e chunking

**Scenario:** La knowledge base viene caricata da un file di testo (`azienda_documenti.txt`) che rappresenta il manuale aziendale completo di TechSphere S.r.l.

**Differenze rispetto all'Esempio 1:**
| Feature | Esempio 1 | Esempio 2 |
|---|---|---|
| Fonte dati | Lista di frasi hard-coded | File di testo esterno |
| Pre-processing | Nessuno | Cleaning + Chunking |
| Granularità | Frase singola | Chunk con overlap |
| Metadati | Solo ID | ID + sorgente + sezione |

**Flusso:**
```
File .txt ──▶ Lettura ──▶ Cleaning ──▶ Chunking ──▶ Embedding ──▶ ChromaDB
                                                                       │
Query utente ──▶ Embedding ─────────────────────────▶ Similarity Search
                                                                       │
                                                               Top-K Chunks
                                                                       │
                                                         Prompt + Contesto
                                                                       │
                                                              Ollama LLM
                                                                       │
                                                                  Risposta
```

### 2.1 — Lettura del file

In [11]:
import os

FILE_PATH = "azienda_documenti.txt"

# Leggi il file
with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"✅ File letto: '{FILE_PATH}'")
print(f"   Lunghezza totale: {len(raw_text)} caratteri")
print(f"   Numero di righe: {raw_text.count(chr(10))}")
print("\n--- Anteprima (prime 500 caratteri) ---")
print(raw_text[:500])

✅ File letto: 'azienda_documenti.txt'
   Lunghezza totale: 7832 caratteri
   Numero di righe: 162

--- Anteprima (prime 500 caratteri) ---
MANUALE AZIENDALE - TECHSPHERE S.R.L.
Versione 3.2 | Aggiornamento: Gennaio 2024

SEZIONE 1 - PROFILO AZIENDALE
------------------------------

TechSphere S.r.l. è una società italiana fondata nel 2010 con sede principale a Milano, in Via della Innovazione 42.
L'azienda opera nel settore dello sviluppo software, della consulenza IT e dei servizi cloud.
Il codice fiscale e partita IVA della società è IT09876543210.
TechSphere conta attualmente 350 dipenden


### 2.2 — Cleaning del testo

Il cleaning rimuove elementi rumorosi che potrebbero degradare la qualità degli embedding:
- Righe di separazione (`===`, `---`)
- Spazi multipli e ritorni a capo eccessivi
- Caratteri speciali non rilevanti

In [12]:
import re

def clean_text(text: str) -> str:
    """
    Pulisce il testo grezzo rimuovendo elementi di rumore.
    
    Steps:
      1. Rimuove le righe di separazione (=====, -----)
      2. Rimuove spazi bianchi a inizio/fine di ogni riga
      3. Normalizza gli a capo multipli (max 2 consecutivi)
      4. Rimuove spazi multipli interni
    """
    # 1. Rimuove le righe di soli simboli (=, -, *)
    text = re.sub(r'^[=\-\*]{3,}\s*$', '', text, flags=re.MULTILINE)

    # 2. Rimuove spazi a inizio/fine riga
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(lines)

    # 3. Normalizza gli a capo multipli (riduce a max 2 newline consecutive)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # 4. Rimuove spazi multipli interni alla riga
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()


cleaned_text = clean_text(raw_text)

print("✅ Testo pulito.")
print(f"   Lunghezza originale: {len(raw_text)} caratteri")
print(f"   Lunghezza dopo cleaning: {len(cleaned_text)} caratteri")
print(f"   Riduzione: {(1 - len(cleaned_text)/len(raw_text))*100:.1f}%")
print("\n--- Anteprima testo pulito (prime 500 caratteri) ---")
print(cleaned_text[:500])

✅ Testo pulito.
   Lunghezza originale: 7832 caratteri
   Lunghezza dopo cleaning: 7492 caratteri
   Riduzione: 4.3%

--- Anteprima testo pulito (prime 500 caratteri) ---
MANUALE AZIENDALE - TECHSPHERE S.R.L.
Versione 3.2 | Aggiornamento: Gennaio 2024

SEZIONE 1 - PROFILO AZIENDALE

TechSphere S.r.l. è una società italiana fondata nel 2010 con sede principale a Milano, in Via della Innovazione 42.
L'azienda opera nel settore dello sviluppo software, della consulenza IT e dei servizi cloud.
Il codice fiscale e partita IVA della società è IT09876543210.
TechSphere conta attualmente 350 dipendenti distribuiti in quattro sedi: Milano (sede principale), Roma, Torino e


### 2.3 — Chunking del testo

Il **chunking** divide il testo in pezzi (chunk) di dimensione controllata.

**Perché è necessario?**
- Gli LLM hanno un context window limitato
- Chunk più piccoli migliorano la precisione del retrieval
- Chunk con **overlap** evitano di spezzare concetti importanti

```
Testo originale:
[........................................]

Chunking con chunk_size=200, overlap=50:
[--------chunk1--------]
                  [--------chunk2--------]
                                    [--------chunk3--------]
                  |<--- overlap --->|
```

In [ ]:
def split_into_chunks(
    text: str,
    chunk_size: int = 400,
    overlap: int = 80,
    separator: str = "\n\n"
) -> list[dict]:
    """
    Divide il testo in chunk con overlap.
    
    Strategia:
      1. Prima divide per paragrafi (separator)
      2. Poi unisce i paragrafi in chunk fino a chunk_size
      3. Aggiunge un overlap tra chunk consecutivi
    
    Args:
        text: il testo da dividere
        chunk_size: dimensione massima di ogni chunk (in caratteri)
        overlap: numero di caratteri sovrapposti tra chunk consecutivi
        separator: separatore tra paragrafi
    
    Returns:
        lista di dizionari con 'text' e 'metadata'
    """
    # Divide il testo in paragrafi
    paragraphs = [p.strip() for p in text.split(separator) if p.strip()]

    chunks = []
    current_chunk = ""
    chunk_index = 0
    
    current_section = ""
    next_section = ""
    for para in paragraphs:
        if para.startswith("SEZIONE"): # Memorizza la prossima sezione in cui sei
            next_section = para[:]

        # Se aggiungere il paragrafo supererebbe il chunk_size, salva il chunk corrente
        if (len(current_chunk) + len(para) > chunk_size and current_chunk) or (next_section != current_section):
            chunks.append({
                "text": current_chunk.strip(),
                "metadata": {
                    "source": FILE_PATH,
                    "chunk_index": chunk_index,
                    "char_count": len(current_chunk.strip()),
                    "section": current_section
                }
            })

            if(next_section != current_section):    # Se sei in una nuova sezione inizia un nuovo chunk
                current_section = next_section
                current_chunk = ""
                chunk_index += 1
                continue

            # Overlap: prendi gli ultimi 'overlap' caratteri del chunk corrente
            current_chunk = current_chunk[-overlap:] + " " + para
            chunk_index += 1
            
        else:
            current_chunk = (current_chunk + " " + para).strip() if current_chunk else para

    # Aggiungi l'ultimo chunk se non vuoto
    if current_chunk.strip():
        chunks.append({
            "text": current_chunk.strip(),
            "metadata": {
                "source": FILE_PATH,
                "chunk_index": chunk_index,
                "char_count": len(current_chunk.strip()),
                "section": current_section
            }
        })

    return chunks


chunks = split_into_chunks(cleaned_text, chunk_size=400, overlap=80)

print(f"✅ Testo diviso in {len(chunks)} chunk.")
print(f"   Dimensione media chunk: {sum(c['metadata']['char_count'] for c in chunks) // len(chunks)} caratteri")
print(f"   Dimensione min: {min(c['metadata']['char_count'] for c in chunks)} caratteri")
print(f"   Dimensione max: {max(c['metadata']['char_count'] for c in chunks)} caratteri")
print("\n--- Esempio Chunk #0 ---")
print(chunks[0]['text'])
print("\n--- Esempio Chunk #1 ---")
print(chunks[1]['text'])

✅ Testo diviso in 21 chunk.
   Dimensione media chunk: 386 caratteri
   Dimensione min: 58 caratteri
   Dimensione max: 721 caratteri

--- Esempio Chunk #0 ---
MANUALE AZIENDALE - TECHSPHERE S.R.L.
Versione 3.2 | Aggiornamento: Gennaio 2024

--- Esempio Chunk #1 ---
TechSphere S.r.l. è una società italiana fondata nel 2010 con sede principale a Milano, in Via della Innovazione 42.
L'azienda opera nel settore dello sviluppo software, della consulenza IT e dei servizi cloud.
Il codice fiscale e partita IVA della società è IT09876543210.
TechSphere conta attualmente 350 dipendenti distribuiti in quattro sedi: Milano (sede principale), Roma, Torino e Bologna.
La missione aziendale è "Accelerare la trasformazione digitale delle imprese attraverso tecnologie innovative e sostenibili."
Il fatturato annuo per il 2023 è stato di 42 milioni di euro, con una crescita del 18% rispetto all'anno precedente.


In [ ]:
chunks

[{'text': 'MANUALE AZIENDALE - TECHSPHERE S.R.L.\nVersione 3.2 | Aggiornamento: Gennaio 2024',
  'metadata': {'source': 'azienda_documenti.txt',
   'chunk_index': 0,
   'char_count': 80,
   'section': ''}},
 {'text': 'TechSphere S.r.l. è una società italiana fondata nel 2010 con sede principale a Milano, in Via della Innovazione 42.\nL\'azienda opera nel settore dello sviluppo software, della consulenza IT e dei servizi cloud.\nIl codice fiscale e partita IVA della società è IT09876543210.\nTechSphere conta attualmente 350 dipendenti distribuiti in quattro sedi: Milano (sede principale), Roma, Torino e Bologna.\nLa missione aziendale è "Accelerare la trasformazione digitale delle imprese attraverso tecnologie innovative e sostenibili."\nIl fatturato annuo per il 2023 è stato di 42 milioni di euro, con una crescita del 18% rispetto all\'anno precedente.',
  'metadata': {'source': 'azienda_documenti.txt',
   'chunk_index': 1,
   'char_count': 639,
   'section': 'SEZIONE 1 - PROFILO AZIEN

### 2.4 — Indicizzazione su ChromaDB con metadati

In [14]:
import chromadb
from chromadb.utils import embedding_functions

# Client ChromaDB in-memory (separato dall'esempio 1)
client2 = chromadb.Client()

# Stessa funzione di embedding multilingua
embedding_fn2 = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# Crea la collection per i documenti aziendali
collection_name = "techsphere_manuale"

if any(c.name == collection_name for c in client2.list_collections()):
    client2.delete_collection(collection_name)  # Se già esiste la cancello

collection2 = client2.create_collection(
    name=collection_name,
    embedding_function=embedding_fn2,
    metadata={"hnsw:space": "cosine"}
)

# Prepara i dati per ChromaDB
documents = [chunk['text'] for chunk in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [chunk['metadata'] for chunk in chunks]

# Indicizza tutti i chunk (con metadati)
collection2.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

print(f"✅ Collection '{collection2.name}' creata.")
print(f"   Chunk indicizzati: {collection2.count()}")

✅ Collection 'techsphere_manuale' creata.
   Chunk indicizzati: 21


### 2.5 — Funzione di Retrieval avanzato con metadati

In [15]:
def retrieve_advanced(
    query: str,
    n_results: int = 3
) -> list[dict]:
    """
    Recupera i chunk più rilevanti con i relativi metadati e score.
    
    Args:
        query: la domanda dell'utente
        n_results: numero di chunk da recuperare
    
    Returns:
        lista di dizionari con 'text', 'metadata' e 'distance'
    """
    results = collection2.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    # Zip documenti, metadati e distanze in una lista di dizionari
    retrieved = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        retrieved.append({
            "text": doc,
            "metadata": meta,
            "similarity_score": round(1 - dist, 4)  # converti distanza in similarity
        })

    return retrieved


# Test del retrieval avanzato
test_query = "Quali sono i benefit aziendali?"
results = retrieve_advanced(test_query)

print(f"Query: '{test_query}'")
print(f"\nChunk recuperati ({len(results)}):")
for i, r in enumerate(results, 1):
    print(f"\n  [{i}] Similarity Score: {r['similarity_score']} | Chunk #{r['metadata']['chunk_index']}")
    print(f"       Fonte: {r['metadata']['source']}")
    print(f"       Testo: {r['text'][:200]}...")

Query: 'Quali sono i benefit aziendali?'

Chunk recuperati (3):

  [1] Similarity Score: 0.5081 | Chunk #10
       Fonte: azienda_documenti.txt
       Testo: mensilmente le "Tech Talks": sessioni interne di condivisione delle conoscenze. 4.4 BENEFIT AZIENDALI
- Assicurazione sanitaria integrativa per il dipendente e i familiari a carico
- Ticket restaurant...

  [2] Similarity Score: 0.4528 | Chunk #11
       Fonte: azienda_documenti.txt
       Testo: 5.1 CLIENTI PRINCIPALI
TechSphere serve oltre 500 clienti attivi, principalmente PMI e grandi imprese nei settori:
- Manufacturing e industria (35% del portafoglio clienti)
- Retail e distribuzione (2...

  [3] Similarity Score: 0.4338 | Chunk #18
       Fonte: azienda_documenti.txt
       Testo: a.
I tempi medi di selezione sono di 3-4 settimane dall'invio della candidatura. 7.4 VALUTAZIONE DELLE PERFORMANCE
La valutazione delle performance avviene due volte l'anno (giugno e dicembre).
Il sis...


### 2.6 — Generation con Ollama e contesto strutturato

In [16]:
import ollama

OLLAMA_MODEL = "llama3.2"


def generate_answer_advanced(
    query: str,
    retrieved_chunks: list[dict]
) -> str:
    """
    Genera una risposta usando Ollama con il contesto dei chunk recuperati.
    Usa un prompt strutturato con system message separato.
    
    Args:
        query: la domanda dell'utente
        retrieved_chunks: lista di chunk con testo e metadati
    
    Returns:
        la risposta generata dall'LLM
    """
    # Costruisce il contesto con riferimenti ai chunk
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[Fonte {i} {chunk['metadata']['section']}\n{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    # System message: definisce il ruolo dell'assistente
    system_message = (
        "Sei HR Bot, l'assistente virtuale aziendale di TechSphere S.r.l. "
        "Il tuo compito è rispondere alle domande dei dipendenti sulle policy, benefit e procedure aziendali. "
        "Basati ESCLUSIVAMENTE sulle informazioni fornite nel contesto. "
        "Alla fine delle risposta indica la sezione che hai seguito per dare la risposta"
        "Se l'informazione non è nel contesto, rispondi: 'Non ho informazioni su questo argomento. "
        "Ti suggerisco di contattare l'ufficio HR a hr@techsphere.it'. "
        "Rispondi sempre in italiano, in modo professionale e conciso."
    )

    # User message: contesto + domanda
    user_message = f"""CONTESTO DAL MANUALE AZIENDALE:

{context}

---
DOMANDA DEL DIPENDENTE: {query}"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ]
    )
    return response['message']['content']


print("✅ Funzione generate_answer_advanced definita.")

✅ Funzione generate_answer_advanced definita.


### 2.7 — Pipeline RAG Avanzata completa

In [17]:
def rag_chatbot_advanced(
    query: str,
    n_results: int = 6,
    verbose: bool = True
) -> str:
    """
    Pipeline RAG avanzata completa: retrieve (con score) + generate.
    
    Args:
        query: la domanda dell'utente
        n_results: numero di chunk da recuperare
        verbose: se True, mostra i chunk recuperati con score
    
    Returns:
        la risposta finale del chatbot
    """
    print(f"\n{'='*65}")
    print(f"📩 Domanda: {query}")
    print(f"{'='*65}")

    # Step 1: RETRIEVE (con metadati e similarity score)
    retrieved = retrieve_advanced(query, n_results=n_results)

    if verbose:
        print(f"\n🔍 Chunk recuperati ({len(retrieved)}):")
        for i, r in enumerate(retrieved, 1):
            score = r['similarity_score']
            bar = "█" * int(score * 20)  # barra visuale dello score
            print(f"   [{i}] Score: {score:.4f} {bar}")
            print(f"       Chunk #{r['metadata']['chunk_index']} | {r['text'][:100]}...")

    # Step 2: GENERATE
    print("\n⏳ Generazione risposta in corso...")
    answer = generate_answer_advanced(query, retrieved)

    print(f"\n🤖 HR Bot risponde:\n{answer}")
    print(f"{'='*65}\n")

    return answer

In [18]:
# 🧪 Test con alcune domande al manuale aziendale completo
domande_avanzate = [
    "Quali sono i benefit aziendali di TechSphere?",
    "Come funziona il processo di onboarding per i nuovi dipendenti?",
    "Con quali aziende ha partnership TechSphere?",
    "Come viene effettuata la valutazione delle performance?",
    "Dove si trovano le sedi di TechSphere?",
]

for domanda in domande_avanzate:
    rag_chatbot_advanced(domanda, verbose=False)


📩 Domanda: Quali sono i benefit aziendali di TechSphere?

⏳ Generazione risposta in corso...

🤖 HR Bot risponde:
Sezione seguita: SEZIONE 4 - POLITICHE HR E BENEFIT

I benefici aziendali di TechSphere sono i seguenti:

- Integrazione aziendale fino all'80% della retribuzione per i primi 6 mesi.
- Budget annuo di formazione di 1.500 euro per ogni dipendente.
- Accessi gratuiti alle piattaforme Coursera, Udemy Business e Pluralsight.
- Partecipazione al "TechSphere Academy Program", un percorso di 6 mesi per i neoassunti.
- "Tech Talks", sessioni interne di condivisione delle conoscenze tenute mensilmente.


📩 Domanda: Come funziona il processo di onboarding per i nuovi dipendenti?

⏳ Generazione risposta in corso...

🤖 HR Bot risponde:
Seguo la Fonte 1 SEZIONE 7 - PROCESSI INTERNI

Il processo di onboarding dura 4 settimane e prevede le seguenti fasi:

*   Settimana 1: orientamento aziendale, setup strumenti, introduzione ai team
*   Settimana 2: formazione sui prodotti e servizi TechS

In [ ]:
#CAMBIO MODELLO
OLLAMA_MODEL = "gpt-oss:20b"

# 🧪 Test con alcune domande al manuale aziendale completo
domande_avanzate = [
    "Quali sono i benefit aziendali di TechSphere?",
    "Come funziona il processo di onboarding per i nuovi dipendenti?",
    "Con quali aziende ha partnership TechSphere?",
    "Come viene effettuata la valutazione delle performance?",
    "Dove si trovano le sedi di TechSphere?",
]

for domanda in domande_avanzate:
    rag_chatbot_advanced(domanda, verbose=False)


📩 Domanda: Quali sono i benefit aziendali di TechSphere?

⏳ Generazione risposta in corso...

🤖 HR Bot risponde:
I benefit offerti da TechSphere includono:

| Benefit | Dettaglio |
|---------|-----------|
| **Formazione** | Ogni dipendente ha un budget annuale di 1.500 € per corsi di aggiornamento. L’azienda fornisce accesso gratuito alle piattaforme Coursera, Udemy Business e Pluralsight. |
| **Programma “TechSphere Academy”** | Percorso interno di 6 mesi per i neoassunti, che comprende formazione pratica e mentoring. |
| **Tech Talks** | Sessioni interne mensili di condivisione delle conoscenze tra i team. |
| **Integrazione aziendale** | Copertura di fino all’80 % della retribuzione per i primi 6 mesi di assunzione (es. bonus o incentivi). |

Queste informazioni sono state estratte dalla **Fonte 5 – Sezione 4 “Politiche HR e Benefit”** del manuale aziendale.


📩 Domanda: Come funziona il processo di onboarding per i nuovi dipendenti?

⏳ Generazione risposta in corso...

🤖 HR Bot ri

---
## 🎯 Confronto tra i due approcci

| Aspetto | Esempio 1 (Semplice) | Esempio 2 (Avanzato) |
|---|---|---|
| **Fonte dati** | Lista di frasi hard-coded | File `.txt` esterno |
| **Pre-processing** | Nessuno | Cleaning + Chunking con overlap |
| **Granularità** | Frase singola (~80 char) | Chunk da ~400 char con 80 char overlap |
| **Metadati** | Solo `id` | `source`, `chunk_index`, `char_count` |
| **Similarity score** | Non esposto | Visibile nel retrieval |
| **Prompt** | Singolo messaggio user | System message + user message |
| **Scalabilità** | Bassa (lista statica) | Alta (funziona con qualsiasi file) |

## 📚 Concetti chiave del RAG

1. **Embedding**: trasformazione del testo in un vettore numerico che cattura il significato semantico
2. **Vector Database**: database specializzato per la ricerca di similarità tra vettori
3. **Similarity Search**: trovare i vettori più vicini alla query usando distanza coseno (o euclidea)
4. **Chunking**: dividere documenti lunghi in pezzi gestibili, con overlap per non perdere contesto
5. **Augmented Generation**: il LLM genera la risposta usando il contesto recuperato, non la sola memoria interna
6. **Grounding**: ancorare le risposte del LLM a fatti verificabili (riduce le allucinazioni)

## 🔗 Risorse per approfondire
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [Sentence Transformers](https://www.sbert.net/)
- [Ollama Models](https://ollama.ai/library)
- [RAG Paper originale (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401)

---
# 🖥️ ESEMPIO 3 — RAG con Interfaccia Grafica (Gradio)
## Chatbot aziendale con UI interattiva

**Gradio** permette di costruire interfacce web direttamente nel notebook con poche righe di codice.
L'UI viene servita in locale (o opzionalmente esposta tramite un link pubblico temporaneo).

**Cosa aggiunge rispetto agli esempi precedenti:**
- Campo di testo per inserire la domanda
- Cronologia della conversazione (stile chat)
- Pannello laterale con i chunk recuperati e i loro score
- Slider per regolare il numero di chunk da recuperare (n_results)
- Bottone per pulire la chat

```
┌─────────────────────────────────────────────────────────┐
│  HR Bot — TechSphere                       [Pulisci]   │
│  n_results: ────●──── 3                               │
├──────────────────────────┬──────────────────────────── ┤
│  💬 Chat                 │  🔍 Chunk recuperati        │
│                          │                             │
│  👤 Quante ferie ho?     │  [1] score: 0.91            │
│  🤖 Hai 26 giorni...     │      ████████░░ Sezione HR  │
│                          │      Testo: ...             │
│  [Scrivi qui...] [Invia] │  [2] score: 0.84 ...        │
└──────────────────────────┴─────────────────────────────┘
```


In [ ]:
# Installa Gradio (se non già presente nel virtualenv)
%pip install gradio


6794.09s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 3.1 — Setup: riuso di `collection2`, `retrieve_advanced` e `generate_answer_advanced`

L'Esempio 3 **non ridefinisce** né la knowledge base né le funzioni RAG:
riusa direttamente ciò che è già stato creato nell'Esempio 2.

| Risorsa | Definita in | Riusata nell'Esempio 3 |
|---|---|---|
| `collection2` | Cella 2.4 | ✅ come knowledge base |
| `retrieve_advanced()` | Cella 2.5 | ✅ per il retrieval |
| `generate_answer_advanced()` | Cella 2.6 | ✅ per la generation |
| `OLLAMA_MODEL` | Cella 2.6 | ✅ stesso modello |

> ⚠️ **Prerequisito**: esegui prima tutte le celle dell'Esempio 2 (2.1 → 2.6)
> in modo che `collection2`, `retrieve_advanced` e `generate_answer_advanced` siano in memoria.

In [19]:
# Verifica che le risorse dell'Esempio 2 siano disponibili in memoria
try:
    assert collection2 is not None
    assert callable(retrieve_advanced)
    assert callable(generate_answer_advanced)
    print(f"✅ collection2 pronta ({collection2.count()} chunk indicizzati)")
    print(f"✅ retrieve_advanced disponibile")
    print(f"✅ generate_answer_advanced disponibile")
    print(f"✅ Modello: {OLLAMA_MODEL}")
except (NameError, AssertionError) as e:
    print("❌ Risorse dell'Esempio 2 non trovate in memoria.")
    print("   → Esegui prima tutte le celle dell'Esempio 2 (dalla 2.1 alla 2.6) e riprova.")
    raise


✅ collection2 pronta (21 chunk indicizzati)
✅ retrieve_advanced disponibile
✅ generate_answer_advanced disponibile
✅ Modello: llama3.2


### 3.2 — Funzione di callback per Gradio

L'unica funzione nuova è `handle_message()`, che funge da **collante** tra
l'interfaccia Gradio e la pipeline RAG già definita nell'Esempio 2:

```
Gradio UI ──▶ handle_message() ──▶ retrieve_advanced()      (Esempio 2)
                                 ──▶ generate_answer_advanced() (Esempio 2)
                                 ──▶ build_chunks_html()     (rendering HTML)
```


In [20]:
def build_chunks_html(query: str, chunks: list[dict]) -> str:
    """Costruisce l'HTML per il pannello dei chunk — usato solo dalla UI."""
    parts = [
        "<h4 style='margin:0 0 4px 0; font-size:0.9em;'>🔍 Chunk recuperati per:</h4>",
        f"<p style='margin:0 0 10px 0; color:#57606a; font-style:italic; font-size:0.85em;'>{query}</p>",
        "<hr style='margin:0 0 10px 0;'>",
    ]
    for i, c in enumerate(chunks, 1):
        score = c['similarity_score']  # chiave usata da retrieve_advanced()
        bar_pct = int(score * 100)
        section = c['metadata']['section']
        chunk_idx = c['metadata']['chunk_index']
        preview = c['text'][:250] + ('...' if len(c['text']) > 250 else '')
        bar_color = '#3b82d4' if score >= 0.7 else ('#f59e0b' if score >= 0.5 else '#ef4444')
        parts.append(f"""
<div style='margin-bottom:12px; padding:10px; border:1px solid #e5e7eb;
            border-radius:6px; background:#f7f8fa;'>
  <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;'>
    <strong style='font-size:0.85em;'>[{i}] Chunk #{chunk_idx}</strong>
    <span style='font-family:monospace; font-size:0.85em; color:{bar_color};'>{score:.4f}</span>
  </div>
  <div style='background:#e5e7eb; border-radius:4px; height:5px; margin-bottom:6px;'>
    <div style='width:{bar_pct}%; background:{bar_color}; height:5px; border-radius:4px;'></div>
  </div>
  <div style='font-size:0.75em; color:#57606a; margin-bottom:5px;'>📂 {section}</div>
  <div style='font-size:0.82em; line-height:1.5;'>{preview}</div>
</div>""")
    return ''.join(parts)


def handle_message(message: str, history: list, n_results: int):
    """
    Callback Gradio: chiama retrieve_advanced e generate_answer_advanced
    (definite nell'Esempio 2) e aggiorna i due pannelli della UI.
    """
    if not message.strip():
        return history, '', "<p><em>Scrivi una domanda nel campo chat.</em></p>"

    # Riusa retrieve_advanced dell'Esempio 2
    chunks = retrieve_advanced(message, n_results=int(n_results))

    # Riusa generate_answer_advanced dell'Esempio 2
    answer = generate_answer_advanced(message, chunks)

    chunks_html = build_chunks_html(message, chunks)
    history = history + [
        {'role': 'user', 'content': message},
        {'role': 'assistant', 'content': answer}
    ]
    return history, '', chunks_html


print("✅ Callback Gradio definita.")


✅ Callback Gradio definita.


### 3.3 — Interfaccia Gradio

L'interfaccia ha due colonne:
- **Sinistra (3/5)**: finestra di chat con cronologia e campo di input
- **Destra (2/5)**: pannello con i chunk recuperati e i loro similarity score

Lo **slider** in cima controlla quanti chunk vengono passati al modello come contesto.


In [ ]:
OLLAMA_MODEL = 'gpt-oss:20b'

In [21]:
import gradio as gr

# --- Layout Gradio ---
# Le funzioni handle_message e build_chunks_html sono definite nella cella 3.2 qui sopra.
# retrieve_advanced e generate_answer_advanced vengono dall'Esempio 2.

with gr.Blocks(title='HR Bot — TechSphere', theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """# 🤖 HR Bot — TechSphere S.r.l.
        Chatbot aziendale basato su RAG &nbsp;·&nbsp; Powered by ChromaDB + Ollama"""
    )

    with gr.Row():
        n_results_slider = gr.Slider(
            minimum=1, maximum=6, value=3, step=1,
            label='Numero di chunk da recuperare (n_results)',
            scale=2
        )
        gr.Markdown(
            "*Aumenta per dare più contesto al modello. Diminuisci per risposte più precise e veloci.*",
            scale=3
        )

    with gr.Row(equal_height=True):

        # Colonna sinistra: chat
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label='Conversazione con HR Bot',
                height=520,
            )
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder='Es: Quante ferie ho? Come funziona lo smart working?',
                    label='La tua domanda',
                    lines=1,
                    scale=5
                )
                send_btn = gr.Button('Invia ➤', variant='primary', scale=1)
            clear_btn = gr.Button('🗑️  Pulisci chat', variant='secondary')

        # Colonna destra: chunk recuperati
        with gr.Column(scale=2):
            chunks_panel = gr.HTML(
                value="<p style='color:#57606a; padding:16px;'>"
                      "<em>I chunk recuperati appariranno qui dopo la prima domanda.</em></p>",
                label='Chunk recuperati dal Vector DB'
            )

    history_state = gr.State([])

    # Bottone Invia
    send_btn.click(
        fn=handle_message,
        inputs=[msg_input, history_state, n_results_slider],
        outputs=[chatbot, msg_input, chunks_panel]
    ).then(lambda h: h, inputs=[chatbot], outputs=[history_state])

    # Tasto Invio nella textbox
    msg_input.submit(
        fn=handle_message,
        inputs=[msg_input, history_state, n_results_slider],
        outputs=[chatbot, msg_input, chunks_panel]
    ).then(lambda h: h, inputs=[chatbot], outputs=[history_state])

    # Bottone pulisci
    clear_btn.click(
        fn=lambda: ([], [], "<p><em>Chat pulita.</em></p>"),
        outputs=[chatbot, history_state, chunks_panel]
    )


# Avvia l'interfaccia su http://127.0.0.1:7860
# share=True genera un link pubblico temporaneo (72h) — utile per demo
demo.launch(share=False)

C:\Users\A1627apulia\AppData\Local\Temp\ipykernel_20680\1808820110.py:7: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='HR Bot — TechSphere', theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Note su Gradio

| Parametro | Valore default | Descrizione |
|---|---|---|
| `share=False` | locale | L'app gira su `http://127.0.0.1:7860` |
| `share=True` | link pubblico | Gradio genera un URL `xxx.gradio.live` valido 72h |
| `OLLAMA_MODEL_UI` | `'llama3.2'` | Cambia nella cella 3.2 per usare un altro modello |
| `n_results` slider | 1–6 | Chunk passati al modello come contesto |

Per chiudere il server:
```python
demo.close()
```


In [ ]:
chunks[2]

{'text': "L'organigramma di TechSphere è composto dai seguenti dipartimenti:\n- Direzione Generale: guidata dal CEO Marco Ferretti\n- Direzione Tecnica (CTO): Laura Conti supervisiona tutte le attività di sviluppo software\n- Dipartimento Commerciale e Vendite: gestito da Roberto Mancini, con 45 commerciali attivi\n- Dipartimento Risorse Umane (HR): responsabile Giulia Rossi, gestisce assunzioni, formazione e welfare\n- Dipartimento Finance e Amministrazione: CFO Stefano Bianchi\n- Dipartimento Marketing e Comunicazione: responsabile Anna Vitali\n- Dipartimento Ricerca e Sviluppo (R&D): team di 60 ingegneri coordinati da Davide Esposito\n- Dipartimento Customer Success: gestito da Federica Longo, 30 consulenti dedicati ai clienti",
 'metadata': {'source': 'azienda_documenti.txt',
  'chunk_index': 2,
  'char_count': 721,
  'section': 'SEZIONE 2 - STRUTTURA ORGANIZZATIVA'}}

---
# 🦜 ESEMPIO 4 — RAG con LangChain
## Stessa pipeline dell'Esempio 2, riscritta con LangChain

**Obiettivo:** replicare fedelmente il flusso dell'Esempio 2 (file → cleaning → chunking → vector store → retrieval → generation) usando le astrazioni di **LangChain** al posto del codice manuale.

**Differenze rispetto all'Esempio 2:**
| Componente | Esempio 2 (manuale) | Esempio 4 (LangChain) |
|---|---|---|
| Lettura file | `open()` + `f.read()` | `TextLoader` |
| Chunking | funzione `split_into_chunks()` custom | `RecursiveCharacterTextSplitter` |
| Vector Store | ChromaDB SDK diretto | `Chroma` di LangChain (wrapper) |
| Embedding | `SentenceTransformerEmbeddingFunction` | `HuggingFaceEmbeddings` |
| Retrieval | `collection2.query()` manuale | `vectorstore.as_retriever()` |
| Generation | `ollama.chat()` + prompt custom | `ChatOllama` + `ChatPromptTemplate` + LCEL |
| Pipeline | funzione `rag_chatbot_advanced()` | Chain LCEL: `retriever | prompt | llm | parser` |

**Flusso LangChain:**
```
TextLoader ──▶ RecursiveCharacterTextSplitter ──▶ HuggingFaceEmbeddings ──▶ Chroma
                                                                               │
Query utente ──▶ Retriever ─────────────────────────────▶ top-K Documents
                                                                               │
                                                    ChatPromptTemplate + ChatOllama
                                                                               │
                                                                          Risposta
```

### 4.0 — Installazione dipendenze LangChain

```bash
pip install langchain langchain-community langchain-ollama langchain-chroma langchain-huggingface sentence-transformers
```
Oppure esegui la cella qui sotto:

In [ ]:
%pip install -q langchain langchain-community langchain-ollama langchain-chroma langchain-huggingface sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 4.1 — Lettura del file con `TextLoader`

`TextLoader` è il corrispondente LangChain dell'`open()` + `f.read()` dell'Esempio 2.
Restituisce una lista di oggetti `Document`, ognuno con `.page_content` e `.metadata`.

In [22]:
from langchain_community.document_loaders.text import TextLoader


from langchain_core.documents.base import Document


from langchain_community.document_loaders import TextLoader

FILE_PATH_4 = "azienda_documenti.txt"

# TextLoader legge il file e lo avvolge in un oggetto Document
loader: TextLoader = TextLoader(FILE_PATH_4, encoding="utf-8")
raw_docs: list[Document] = loader.load()

print(f"✅ File caricato con TextLoader.")
print(f"   Numero di Document restituiti: {len(raw_docs)}")
print(f"   Lunghezza testo: {len(raw_docs[0].page_content)} caratteri")
print(f"   Metadati automatici: {raw_docs[0].metadata}")
print("\n--- Anteprima (prime 300 caratteri) ---")
print(raw_docs[0].page_content[:300])

C:\Users\A1627apulia\AppData\Local\Temp\ipykernel_20680\605816776.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


✅ File caricato con TextLoader.
   Numero di Document restituiti: 1
   Lunghezza testo: 7832 caratteri
   Metadati automatici: {'source': 'azienda_documenti.txt'}

--- Anteprima (prime 300 caratteri) ---
MANUALE AZIENDALE - TECHSPHERE S.R.L.
Versione 3.2 | Aggiornamento: Gennaio 2024

SEZIONE 1 - PROFILO AZIENDALE
------------------------------

TechSphere S.r.l. è una società italiana fondata nel 2010 con sede principale a Milano, in Via della Innovazione 42


### 4.2 — Chunking con `RecursiveCharacterTextSplitter`

`RecursiveCharacterTextSplitter` è l'equivalente LangChain della funzione `split_into_chunks()` dell'Esempio 2.
Divide il testo provando in sequenza i separatori `["\n\n", "\n", " "]` finché i chunk rientrano in `chunk_size`.
Il parametro `chunk_overlap` corrisponde all'`overlap=80` dell'Esempio 2.

| Parametro | Esempio 2 | Esempio 4 |
|---|---|---|
| `chunk_size` | 400 caratteri | 400 caratteri |
| `overlap` | 80 caratteri | 80 caratteri |
| Separatori | `\n\n` (paragrafi) | `["\n\n", "\n", " "]` (ricorsivo) |

In [23]:
from langchain_core.documents.base import Document


from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", " "],
    length_function=len,
)

# split_documents accetta una lista di Document e restituisce Document splittati
lc_chunks: list[Document] = splitter.split_documents(raw_docs)

print(f"✅ Testo diviso in {len(lc_chunks)} chunk.")
char_counts = [len(c.page_content) for c in lc_chunks]
print(f"   Dimensione media: {sum(char_counts) // len(char_counts)} caratteri")
print(f"   Dimensione min:   {min(char_counts)} caratteri")
print(f"   Dimensione max:   {max(char_counts)} caratteri")
print("\n--- Esempio Chunk #0 ---")
print(lc_chunks[0].page_content)
print(f"Metadati: {lc_chunks[0].metadata}")
print("\n--- Esempio Chunk #1 ---")
print(lc_chunks[1].page_content)
print(f"Metadati: {lc_chunks[1].metadata}")

✅ Testo diviso in 29 chunk.
   Dimensione media: 273 caratteri
   Dimensione min:   73 caratteri
   Dimensione max:   397 caratteri

--- Esempio Chunk #0 ---
MANUALE AZIENDALE - TECHSPHERE S.R.L.
Versione 3.2 | Aggiornamento: Gennaio 2024

SEZIONE 1 - PROFILO AZIENDALE
------------------------------
Metadati: {'source': 'azienda_documenti.txt'}

--- Esempio Chunk #1 ---
TechSphere S.r.l. è una società italiana fondata nel 2010 con sede principale a Milano, in Via della Innovazione 42.
L'azienda opera nel settore dello sviluppo software, della consulenza IT e dei servizi cloud.
Il codice fiscale e partita IVA della società è IT09876543210.
TechSphere conta attualmente 350 dipendenti distribuiti in quattro sedi: Milano (sede principale), Roma, Torino e Bologna.
Metadati: {'source': 'azienda_documenti.txt'}


### 4.3 — Indicizzazione su Chroma con `HuggingFaceEmbeddings`

In LangChain:
- `HuggingFaceEmbeddings` sostituisce `SentenceTransformerEmbeddingFunction` di ChromaDB
- `Chroma.from_documents()` crea il vector store e indicizza i chunk in un'unica chiamata,
  sostituendo `client.create_collection()` + `collection.add()`

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Stesso modello multilingua usato nell'Esempio 2
embedding_model = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# Crea il vector store in-memory e indicizza tutti i chunk in un'unica chiamata
# Equivale a: client2.create_collection() + collection2.add(documents, ids, metadatas)
vectorstore4 = Chroma.from_documents(
    documents=lc_chunks,
    embedding=embedding_model,
    collection_name="techsphere_lc",
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"✅ Vector store LangChain creato.")
print(f"   Chunk indicizzati: {vectorstore4._collection.count()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Vector store LangChain creato.
   Chunk indicizzati: 29


### 4.4 — Retriever

`vectorstore.as_retriever()` restituisce un oggetto `Retriever` standard di LangChain,
equivalente alla funzione `retrieve_advanced()` dell'Esempio 2.
Il parametro `k` corrisponde a `n_results` dell'Esempio 2.

In [25]:
# Retriever con k=3 documenti (equivalente a n_results=3 dell'Esempio 2)
retriever4 = vectorstore4.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# Test del retriever — equivalente di retrieve_advanced(test_query)
test_query = "Quali sono i benefit aziendali?"
retrieved_docs = retriever4.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"\nDocumenti recuperati ({len(retrieved_docs)}):")
for i, doc in enumerate(retrieved_docs, 1):
    fonte = doc.metadata.get('source', 'n/a')
    print(f"\n  [{i}] Fonte: {fonte}")
    print(f"       Testo: {doc.page_content[:200]}...")

Query: 'Quali sono i benefit aziendali?'

Documenti recuperati (3):

  [1] Fonte: azienda_documenti.txt
       Testo: 4.4 BENEFIT AZIENDALI
- Assicurazione sanitaria integrativa per il dipendente e i familiari a carico
- Ticket restaurant del valore di 8 euro al giorno
- Rimborso abbonamento mezzi pubblici fino a 100...

  [2] Fonte: azienda_documenti.txt
       Testo: La missione aziendale è "Accelerare la trasformazione digitale delle imprese attraverso tecnologie innovative e sostenibili."
Il fatturato annuo per il 2023 è stato di 42 milioni di euro, con una cres...

  [3] Fonte: azienda_documenti.txt
       Testo: SEZIONE 5 - CLIENTI E PARTNER
--------------------------------

5.1 CLIENTI PRINCIPALI
TechSphere serve oltre 500 clienti attivi, principalmente PMI e grandi imprese nei settori:
- Manufacturing e ind...


### 4.5 — Chain RAG con LCEL

**LCEL (LangChain Expression Language)** permette di comporre la pipeline RAG con l'operatore `|`.
L'intera pipeline `retrieve → format → prompt → llm → parse` diventa una singola espressione.

```python
chain = {"context": retriever | format_docs, "question": passthrough} | prompt | llm | parser
```

| Componente LCEL | Equivalente Esempio 2 |
|---|---|
| `retriever4` | `retrieve_advanced(query, n_results)` |
| `format_docs` | costruzione di `context_parts` in `generate_answer_advanced()` |
| `ChatPromptTemplate` | `system_message` + `user_message` costruiti a mano |
| `ChatOllama` | `ollama.chat(model=OLLAMA_MODEL, messages=[...])` |
| `StrOutputParser` | `response['message']['content']` |

In [26]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

OLLAMA_MODEL_LC = "llama3.2"  # cambia con il modello che preferisci

# LLM: equivalente di ollama.chat(model=OLLAMA_MODEL, ...)
llm4 = ChatOllama(model=OLLAMA_MODEL_LC, temperature=0)

# Prompt: equivalente di system_message + user_message dell'Esempio 2
prompt4 = ChatPromptTemplate.from_messages([
    ("system",
     "Sei HR Bot, l'assistente virtuale aziendale di TechSphere S.r.l. "
     "Il tuo compito è rispondere alle domande dei dipendenti sulle policy, benefit e procedure aziendali. "
     "Basati ESCLUSIVAMENTE sulle informazioni fornite nel contesto. "
     "Alla fine della risposta indica la sezione del manuale che hai usato. "
     "Se l'informazione non è nel contesto, rispondi: "
     "'Non ho informazioni su questo argomento. Ti suggerisco di contattare hr@techsphere.it'. "
     "Rispondi sempre in italiano, in modo professionale e conciso."),
    ("human",
     "CONTESTO DAL MANUALE AZIENDALE:\n\n{context}\n\n---\nDOMANDA DEL DIPENDENTE: {question}"),
])


def format_docs(docs) -> str:
    """Formatta i Document recuperati in una stringa di contesto.
    Equivalente della costruzione di context_parts in generate_answer_advanced()."""
    return "\n\n".join(
        f"[Fonte {i+1}]\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )


# Chain LCEL — equivalente dell'intera funzione rag_chatbot_advanced() dell'Esempio 2
# Permette di comporre la pipeline con l'operatore "|"
# funziona in modo molto simile a una pipe Unix: l'output di uno step diventa l'input dello step successivo.
rag_chain4 = (
    {"context": retriever4 | format_docs, "question": RunnablePassthrough()}    # l'ouput di retriever4 va in input alla funzione format_docs; RunnablePassthrough passa l'input (quindi la query)
    | prompt4   # il dizionario sopra va in input a prompt4 per formare il prompt
    | llm4      # il prompt viene passato al modello
    | StrOutputParser() #StrOutputParser() estrae il testo dalla risposta
)

print("✅ Chain RAG LangChain definita.")
print(f"   Modello: {OLLAMA_MODEL_LC}")
print(f"   Retriever k={retriever4.search_kwargs['k']}")

✅ Chain RAG LangChain definita.
   Modello: llama3.2
   Retriever k=3


### 4.6 — Test della pipeline

La chain si invoca con `rag_chain4.invoke(query)` — una singola chiamata che esegue
in sequenza retrieval, formatting, generation e parsing.
Usiamo le **stesse domande dell'Esempio 2** per confrontare direttamente i risultati.

In [27]:
from langchain_core.documents.base import Document


def rag_chatbot_lc(query: str, verbose: bool = True) -> str:
    """
    Pipeline RAG LangChain completa.
    Equivalente di rag_chatbot_advanced() dell'Esempio 2,
    ma delegata interamente alla chain LCEL.
    """
    print(f"\n{'='*65}")
    print(f"📩 Domanda: {query}")
    print(f"{'='*65}")

    if verbose:
        # Mostra i chunk recuperati (per confronto con l'Esempio 2)
        docs: list[Document] = retriever4.invoke(query)
        print(f"\n🔍 Chunk recuperati ({len(docs)}):")
        for i, doc in enumerate(docs, 1):
            print(f"   [{i}] {doc.page_content[:120]}...")

    print("\n⏳ Generazione risposta in corso...")
    answer = rag_chain4.invoke(input=query)

    print(f"\n🤖 HR Bot risponde:\n{answer}")
    print(f"{'='*65}\n")
    return answer


# Stesse domande dell'Esempio 2 per confrontare i risultati
domande_lc = [
    "Quali sono i benefit aziendali di TechSphere?",
    "Come funziona il processo di onboarding per i nuovi dipendenti?",
    "Con quali aziende ha partnership TechSphere?",
    "Come viene effettuata la valutazione delle performance?",
    "Dove si trovano le sedi di TechSphere?",
]

for domanda in domande_lc:
    rag_chatbot_lc(domanda, verbose=False)


📩 Domanda: Quali sono i benefit aziendali di TechSphere?

⏳ Generazione risposta in corso...

🤖 HR Bot risponde:
Grazie per avermi posto la domanda! Secondo il manuale aziendale, non sono state fornite informazioni specifiche sui benefit aziendali di TechSphere. Tuttavia, posso informarti che l'azienda offre una gamma completa di prodotti e servizi, e che conta attualmente 350 dipendenti distribuiti in quattro sedi.

Se desideri informazioni sui benefit aziendali, ti suggerisco di contattare l'HR department di TechSphere, che potrà fornirti le informazioni più aggiornate e precise.

Sei libero di chiedere un'altra domanda?


📩 Domanda: Come funziona il processo di onboarding per i nuovi dipendenti?

⏳ Generazione risposta in corso...

🤖 HR Bot risponde:
Il processo di onboarding per i nuovi dipendenti a TechSphere S.r.l. è suddiviso in 4 settimane:

- Settimana 1: orientamento aziendale, setup strumenti, introduzione ai team
- Settimana 2: formazione sui prodotti e servizi TechSphere


---
## 🔄 Confronto Esempio 2 vs Esempio 4

| Aspetto | Esempio 2 (manuale) | Esempio 4 (LangChain) |
|---|---|---|
| **Righe di codice** | ~120 | ~60 |
| **Lettura file** | `open()` + `f.read()` | `TextLoader` |
| **Chunking** | funzione custom con logica sezioni | `RecursiveCharacterTextSplitter` |
| **Embedding** | `SentenceTransformerEmbeddingFunction` (Chroma SDK) | `HuggingFaceEmbeddings` (LangChain) |
| **Vector Store** | `chromadb.Client()` + `create_collection()` + `add()` | `Chroma.from_documents()` |
| **Retrieval** | `collection.query()` + parsing manuale | `vectorstore.as_retriever().invoke()` |
| **Prompt** | f-string con system/user costruita a mano | `ChatPromptTemplate.from_messages()` |
| **LLM call** | `ollama.chat()` + `response['message']['content']` | `ChatOllama` + `StrOutputParser` |
| **Pipeline** | funzione Python con step espliciti | Chain LCEL: `retriever \| prompt \| llm \| parser` |

**Quando usare LangChain:**
- Prototipazione rapida con componenti intercambiabili
- Pipeline complesse (multi-step, memoria, agenti)
- Integrazione con molti LLM/vector store senza cambiare il codice

**Quando usare l'approccio manuale (Esempio 2):**
- Massimo controllo sul chunking (es. suddivisione per sezione del manuale)
- Contesti dove le dipendenze extra non sono desiderate
- Debug fine di problemi specifici di retrieval